<a href="https://colab.research.google.com/github/geethat0126-bit/BIS_5TH_SEM/blob/main/week3_1BM24CS104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import random

# ---------------- USER INPUT ----------------

print("CAMPUS SHUTTLE BUS ROUTE OPTIMIZATION")
print("--------------------------------------")

n = int(input("Enter number of locations (including depot): "))

locations = []

for i in range(n):
    name = input(f"Enter location {i + 1}: ")
    locations.append(name)

print("\nFirst location is considered as DEPOT.")

# Passenger demand
demand = [0]

for i in range(1, n):
    d = int(input(f"Enter passenger demand at {locations[i]}: "))
    demand.append(d)

capacity = int(input("\nEnter bus capacity: "))

# Distance matrix
distance = [[0] * n for _ in range(n)]

print("\nEnter distance between locations (in km):")

for i in range(n):
    for j in range(i + 1, n):
        d = float(input(
            f"Distance from {locations[i]} to {locations[j]}: "
        ))
        distance[i][j] = d
        distance[j][i] = d

# Travel time matrix
time = [[0] * n for _ in range(n)]

print("\nEnter travel time between locations (in minutes):")

for i in range(n):
    for j in range(i + 1, n):
        t = float(input(
            f"Time from {locations[i]} to {locations[j]}: "
        ))
        time[i][j] = t
        time[j][i] = t

# ---------------- ACO PARAMETERS ----------------

ants = 20
iterations = 100
alpha = 1
beta = 2
rho = 0.5
Q = 100

# Initialize pheromone matrix
pheromone = [[1.0] * n for _ in range(n)]


# ---------------- COST FUNCTION ----------------

def calculate_cost(route):

    total_distance = 0
    total_time = 0

    for i in range(len(route) - 1):
        a = route[i]
        b = route[i + 1]

        total_distance += distance[a][b]
        total_time += time[a][b]

    return total_distance, total_time


def route_cost(route):

    d, t = calculate_cost(route)

    return d + t


# ---------------- CONSTRUCT ROUTE ----------------

def construct_route():

    unvisited = set(range(1, n))

    route = [0]
    current = 0
    load = 0

    while unvisited:

        feasible = [
            j for j in unvisited
            if demand[j] <= capacity - load
        ]

        # Return to depot if capacity is insufficient
        if not feasible:

            route.append(0)
            current = 0
            load = 0
            continue

        probabilities = []

        for j in feasible:

            eta = 1 / (distance[current][j] +
                       time[current][j] + 0.001)

            value = (
                pheromone[current][j] ** alpha
            ) * (eta ** beta)

            probabilities.append(value)

        total = sum(probabilities)

        probabilities = [
            p / total for p in probabilities
        ]

        next_location = random.choices(
            feasible,
            weights=probabilities
        )[0]

        route.append(next_location)

        load += demand[next_location]

        unvisited.remove(next_location)

        current = next_location

    route.append(0)

    return route


# ---------------- ACO ALGORITHM ----------------

best_route = None
best_cost = float("inf")

for iteration in range(iterations):

    ant_routes = []

    for ant in range(ants):

        route = construct_route()

        cost = route_cost(route)

        ant_routes.append((route, cost))

        if cost < best_cost:

            best_cost = cost
            best_route = route

    # Pheromone evaporation
    for i in range(n):
        for j in range(n):

            pheromone[i][j] *= (1 - rho)

    # Pheromone update
    for route, cost in ant_routes:

        deposit = Q / cost

        for k in range(len(route) - 1):

            a = route[k]
            b = route[k + 1]

            pheromone[a][b] += deposit
            pheromone[b][a] += deposit


# ---------------- DISPLAY RESULT ----------------

total_distance, total_time = calculate_cost(best_route)

print("\n========== OPTIMIZED SHUTTLE ROUTE ==========")

trip = 1
passengers = 0

print("\nTrip", trip)
print(locations[0], end="")

for i in range(1, len(best_route)):

    node = best_route[i]

    if node == 0:

        print(" ->", locations[0])

        if i < len(best_route) - 1:

            trip += 1
            passengers = 0

            print("\nTrip", trip)
            print(locations[0], end="")

    else:

        print(" ->", locations[node], end="")
        passengers += demand[node]

print("\n\nTotal Distance:", total_distance, "km")
print("Total Travel Time:", total_time, "minutes")
print("Total Passenger Demand:", sum(demand))
print("Bus Capacity:", capacity)
print("Number of Trips:", best_route.count(0) - 1)
print("Minimum Objective Cost:", best_cost)

print("\nAll passenger demands are served.")


CAMPUS SHUTTLE BUS ROUTE OPTIMIZATION
--------------------------------------
Enter number of locations (including depot): 5
Enter location 1: Depot
Enter location 2: Hostel
Enter location 3: Library
Enter location 4: CSE Block
Enter location 5: Parking

First location is considered as DEPOT.
Enter passenger demand at Hostel: 6
Enter passenger demand at Library: 4
Enter passenger demand at CSE Block: 5
Enter passenger demand at Parking: 3

Enter bus capacity: 10

Enter distance between locations (in km):
Distance from Depot to Hostel: 2
Distance from Depot to Library: 4
Distance from Depot to CSE Block: 5
Distance from Depot to Parking: 3
Distance from Hostel to Library: 3
Distance from Hostel to CSE Block: 4
Distance from Hostel to Parking: 5
Distance from Library to CSE Block: 2
Distance from Library to Parking: 4
Distance from CSE Block to Parking: 3

Enter travel time between locations (in minutes):
Time from Depot to Hostel: 5
Time from Depot to Library: 3
Time from Depot to CSE Bl